# Database Governance Quickstart

[!["Open In Colab"](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lakelogic/LakeLogic/blob/main/examples/01_quickstart/02_database_governance.ipynb) 
[![GitHub Repo](https://img.shields.io/badge/GitHub-Repo-blue?logo=github)](https://github.com/lakelogic/LakeLogic/blob/main/examples/01_quickstart/02_database_governance.ipynb)

## 🏢 Business Scenario
Most organizations have critical data locked in traditional **SQL Databases** (ERP, CRM, Legacy apps). To unlock analytics and ML, this data needs to move to a **Modern Data Lakehouse**.

**The Problem**: Database entries are often messy (null emails, invalid IDs, duplicate entries). If you just copy-paste this data into your lake, you're building a "Data Swamp."

**The Solution**: LakeLogic acts as a **Governance Gate**. It extracts the data but automatically redirects messy records to a Quarantine (layer), ensuring only "Gold Standard" data reaches your analytics-ready Silver layer.

## 🎯 What You'll Learn
1. Setting up a "Dirty" source database
2. Defining a **Data Contract** with Quality Rules
3. Using LakeLogic to extract, validate, and protect your Lakehouse
4. Inspecting the Quality split (Good vs. Bad data)

## Set Up

In [ ]:
# SETUP: Install lakelogic if not already present
from pathlib import Path
import os
import importlib.util
import sys
import sqlite3

# --- Package check (works on Colab, Databricks, local) ---
if importlib.util.find_spec("lakelogic") is None:
    print("📦 Installing lakelogic...")
    !pip install lakelogic
    print("✅ lakelogic installed.")
else:
    print("✅ lakelogic is already installed.")

# --- Colab: clone repo so contract files (YAML, etc.) are available ---
if 'google.colab' in sys.modules:
    repo_dir = Path('/content/LakeLogic')
    if not repo_dir.exists():
        print("📂 Cloning LakeLogic repo for example files...")
        !git clone --quiet https://github.com/lakelogic/LakeLogic.git /content/LakeLogic
    os.chdir(repo_dir / 'examples' / '01_quickstart')
    print(f"📍 Working directory set to: {Path.cwd()}")


def find_example_dir(name: str) -> Path:
    cwd = Path.cwd()
    for base in [cwd] + list(cwd.parents):
        for candidate in [
            base / name,
            base / "examples",
            base / "lakelogic" / "examples"
        ]:
            if candidate.exists():
                return candidate
    return cwd


# Global Path Resolver
def get_full_path(filename: str) -> str:
    path_obj = find_example_dir(filename)
    if path_obj.is_dir():
        return str((path_obj / filename).resolve())
    return str(path_obj.resolve())

## 🚀 Step 1: Setup Local Database (with Dirty Data)
We'll use **SQLite** to simulate a source database. We'll intentionally insert 1 good record and 2 records that violate our business rules.

In [ ]:
db_file = 'example.db'
conn = sqlite3.connect(db_file)
c = conn.cursor()
c.execute('CREATE TABLE IF NOT EXISTS users (id INTEGER, name TEXT, email TEXT)')
c.execute('DELETE FROM users')

# 1. Good Record
c.execute('INSERT INTO users VALUES (1, "Alice", "alice@example.com")')

# 2. Bad Record (Invalid Email format)
c.execute('INSERT INTO users VALUES (2, "Bob", "bob-invalid-email-format")')

# 3. Bad Record (Impossible ID)
c.execute('INSERT INTO users VALUES (-99, "Charlie", "charlie@example.com")')

conn.commit()
conn.close()
print("✅ Local database setup complete with 'Dirty Data' (1 Good, 2 Bad)!")

## 📝 Step 2: Use Your Contract
We have a `users_contract.yaml` file that defines our desired schema and **Quality Rules**. LakeLogic will use these rules to filter out the bad records automatically.

In [ ]:
contract_path = get_full_path('users_contract.yaml')
with open(contract_path, 'r') as f:
    print(f"📄 users_contract.yaml content from: {contract_path}")
    print("-----------------------------")
    print(f.read())

## ⚙️ Step 3: Run the Governance Pipeline
LakeLogic validates a dataframe, so we'll extract from SQLite with the standard library and
pass the rows into `DataProcessor.run()`.

We'll use **Named Attribute access** (`result.good`, `result.bad`) for maximum readability.


In [ ]:
from lakelogic import DataProcessor
import sqlite3

# Extract rows from SQLite
conn = sqlite3.connect("example.db")
conn.row_factory = sqlite3.Row
rows = conn.execute("SELECT * FROM users").fetchall()
conn.close()
data = [dict(row) for row in rows]

# Initialize processor with resolved contract path
contract_path = get_full_path('users_contract.yaml')
processor = DataProcessor(contract=contract_path)

# Run the validation pipeline
result = processor.run(data)

print(f"Run Complete via {processor.engine_name} engine!")
print(f"   Records Validated: {len(result.good)}")
print(f"   Records Quarantined: {len(result.bad)}")

## 🎉 Inspect the Results
Let's see the power of LakeLogic: separating the "clean" data from the "corrupt" data.

In [ ]:
print("🚨 QUARANTINE (Rejected DB Rows):")
display(result.bad)

print("\n🏆 CLEAN SILVER DATA (Analytics Ready):")
display(result.good)

## 🏁 Summary
Instead of writing complex validation scripts for every table, you defined a **Data Contract**. LakeLogic ensured that only Alice reached the Lakehouse, while Bob and Charlie were safely isolated for review.

This is **Shift-Left Security** for your Data Lakehouse.